# NFL dedicated models (v1)

These notebooks live in `ml/notebooks/v1/`. Each one is the same pipeline: inspect FEATURES,
walk-forward 2023–24 / 2025, register on `NFL_PROD_DB.ML`, write a pred table.
Do not promote a version because it exists. Do not add FEATURES or ML to agents.

| Model | Family | Task | Label | Notebook |
|---|---|---|---|---|
| `NFL_GAME_TOTAL` | game | regression | `label_total` | `nfl_game_total.ipynb` |
| `NFL_GAME_HOME_POINTS` | game | regression | `label_home_points` | `nfl_game_home_points.ipynb` |
| `NFL_GAME_AWAY_POINTS` | game | regression | `label_away_points` | `nfl_game_away_points.ipynb` |
| `NFL_GAME_MARGIN` | game | regression | `label_home_margin` | `nfl_game_margin.ipynb` |
| `NFL_GAME_HOME_NET_PASS` | game | regression | `label_home_net_pass` | `nfl_game_home_net_pass.ipynb` |
| `NFL_GAME_AWAY_NET_PASS` | game | regression | `label_away_net_pass` | `nfl_game_away_net_pass.ipynb` |
| `NFL_GAME_HOME_RUSH` | game | regression | `label_home_rush` | `nfl_game_home_rush.ipynb` |
| `NFL_GAME_AWAY_RUSH` | game | regression | `label_away_rush` | `nfl_game_away_rush.ipynb` |
| `NFL_PLAYER_PASSING_YARDS` | player | regression | `label_passing_yards` | `nfl_player_passing_yards.ipynb` |
| `NFL_PLAYER_RUSHING_YARDS` | player | regression | `label_rushing_yards` | `nfl_player_rushing_yards.ipynb` |
| `NFL_PLAYER_RECEIVING_YARDS` | player | regression | `label_receiving_yards` | `nfl_player_receiving_yards.ipynb` |
| `NFL_PLAYER_RECEPTIONS` | player | regression | `label_receptions` | `nfl_player_receptions.ipynb` |
| `NFL_PLAYER_PASSING_TDS` | player | regression | `label_passing_tds` | `nfl_player_passing_tds.ipynb` |
| `NFL_PLAYER_ANYTIME_TD` | player | classification | `label_anytime_td` | `nfl_player_anytime_td.ipynb` |

Deferred: first TD (needs play-level order), CLV vs close (2023–25 closes missing),
`feat_player_prop_train` as a later FEATURES grain.

## Packages

`uv pip` against the Snowflake-managed PyPI artifact repo (no Anaconda).
Rerun this after a notebook-service restart. If it cannot reach a repo, attach
the Snowflake PyPI artifact repository on the notebook service — do not point
this at public pypi.org.

In [ ]:
!uv pip install scikit-learn snowflake-ml-python

import importlib.metadata as md

print("sklearn", md.version("scikit-learn"))
print("snowflake-ml-python", md.version("snowflake-ml-python"))

## Session and import

Kernel session via `get_active_session()`. Add `ml/` to `sys.path` so this
notebook can import the same module the laptop `uv run` uses.

In [ ]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for cand in (here, *here.parents):
    if (cand / "weekend_warriors_ml" / "pipeline.py").exists():
        if str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
        break
else:
    raise FileNotFoundError(
        "weekend_warriors_ml not found. Put the repo ml/ folder in this Workspace."
    )

from snowflake.snowpark.context import get_active_session
from weekend_warriors_ml.specs import get_spec
from weekend_warriors_ml.pipeline import (
    cook,
    fit,
    inspect,
    log_experiment,
    register,
    score_batch,
    score_local,
)

SPEC = get_spec("NFL_GAME_TOTAL")
session = get_active_session()
print(SPEC.name, SPEC.task, len(SPEC.feature_columns), "features")
print(session.get_current_role(), session.get_current_warehouse())

In [ ]:
from weekend_warriors_ml.specs import SPECS

for name, spec in SPECS.items():
    print(name, spec.family, spec.task, spec.label_column, spec.notebook)

## Cook one model

Change the name, then run. Do not loop all 14 in one cell on the first pass.

In [ ]:
MODEL = "NFL_GAME_TOTAL"
cook(session, MODEL)